In [ ]:
orders = data["orders"]
paid = data["paid"]

st.subheader("Order Volume & Growth")

total_orders = len(orders)
cancel_rate = orders["Is Cancelled"].mean() * 100 if "Is Cancelled" in orders else 0
cod_rate = (orders["Payment Type"] == "COD").mean() * 100 if "Payment Type" in orders else 0
date_span = (orders["Created at"].max() - orders["Created at"].min()).days

c1, c2, c3, c4 = st.columns(4)
with c1:
    kpi_card("Total Orders", f"{total_orders:,}")
with c2:
    kpi_card("Cancellation Rate", f"{cancel_rate:.2f}%", positive=False)
with c3:
    kpi_card("COD Share", f"{cod_rate:.1f}%")
with c4:
    kpi_card("Date Span", f"{date_span:,} days")

st.markdown("###")

monthly = orders.groupby("Order Month").size().reset_index(name="Orders")
monthly["Order Month"] = monthly["Order Month"].astype(str)
fig = px.line(monthly, x="Order Month", y="Orders", markers=True,
               title="Monthly Order Volume")
fig.update_traces(line_color=ACCENT1, fill="tozeroy", fillcolor="rgba(91,141,239,0.10)")
st.plotly_chart(fig, width='stretch')

In [ ]:
# Peak time analysis + Month-over-Month growth
col1, col2 = st.columns(2)

with col1:
    if "Order Hour" in orders.columns:
        st.markdown("**Peak Order Time**")
        view = st.radio(
            "View", ["Generalized (by hour)", "By day of week"],
            horizontal=True, key="peak_time_view", label_visibility="collapsed",
        )

        if view == "Generalized (by hour)":
            hourly = orders.groupby("Order Hour").size().reset_index(name="Orders")
            fig = px.bar(hourly, x="Order Hour", y="Orders",
                          title="Peak Order Time (All Days Combined)",
                          color_discrete_sequence=[ACCENT1])
            fig.update_layout(xaxis_title="Hour of Day", xaxis=dict(dtick=1))
            st.plotly_chart(fig, width='stretch')
        elif "Order DOW" in orders.columns:
            dow_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
            heat = orders.groupby(["Order DOW", "Order Hour"]).size().reset_index(name="Orders")
            heat_pivot = heat.pivot(index="Order DOW", columns="Order Hour", values="Orders").reindex(dow_order)
            fig = px.imshow(heat_pivot, aspect="auto", color_continuous_scale="Tealrose",
                             title="Peak Order Time (Day vs Hour)")
            st.plotly_chart(fig, width='stretch')

with col2:
    # MoM growth (skip pre-launch months with too few orders to give a
    # meaningful base -- otherwise early ramp-up shows as a 1000s-of-% spike)
    growth = monthly[monthly["Orders"] >= 50].copy()
    growth["Growth %"] = growth["Orders"].pct_change() * 100
    growth = growth.dropna(subset=["Growth %"])
    colors = [SUCCESS if v >= 0 else DANGER for v in growth["Growth %"]]
    fig = go.Figure(go.Bar(x=growth["Order Month"], y=growth["Growth %"], marker_color=colors))
    fig.update_layout(title="Month-over-Month Order Growth (%)")
    st.plotly_chart(fig, width='stretch')
    st.caption("Excludes the earliest pre-launch months (under 50 orders), which would otherwise "
                "show as misleading spikes due to a near-zero base.")

In [ ]:
st.subheader("Geographic Demand")

geo_cols = {"Shipping City", "Shipping Province", "Shipping Zip"}
if geo_cols.issubset(orders.columns):
    fcol1, fcol2 = st.columns(2)

    with fcol1:
        state_counts = orders["Shipping Province"].value_counts()
        state_options = ["All States"] + state_counts.index.tolist()
        sel_state = st.selectbox("Drill into State", state_options, key="geo_state")

    geo_scope = orders if sel_state == "All States" else orders[orders["Shipping Province"] == sel_state]

    with fcol2:
        if sel_state == "All States":
            st.selectbox("Drill into City", ["All Cities"], key="geo_city", disabled=True)
            sel_city = "All Cities"
        else:
            city_counts = geo_scope["Shipping City"].value_counts()
            city_options = ["All Cities"] + city_counts.index.tolist()
            sel_city = st.selectbox("Drill into City", city_options, key="geo_city")

    if sel_city != "All Cities":
        geo_scope = geo_scope[geo_scope["Shipping City"] == sel_city]

    if sel_state == "All States":
        # Top-level view: states, cities, pincodes across all of India
        col1, col2, col3 = st.columns(3)
        with col1:
            top_states = state_counts.head(10).reset_index()
            top_states.columns = ["State", "Orders"]
            fig = px.bar(top_states, x="Orders", y="State", orientation="h",
                          title="Top 10 States by Orders", color_discrete_sequence=[PALETTE[2]])
            fig.update_layout(yaxis=dict(autorange="reversed"))
            st.plotly_chart(fig, width='stretch')
        with col2:
            top_cities = geo_scope["Shipping City"].value_counts().head(10).reset_index()
            top_cities.columns = ["City", "Orders"]
            fig = px.bar(top_cities, x="Orders", y="City", orientation="h",
                          title="Top 10 Cities by Orders", color_discrete_sequence=[ACCENT2])
            fig.update_layout(yaxis=dict(autorange="reversed"))
            st.plotly_chart(fig, width='stretch')
        with col3:
            top_pin = geo_scope["Shipping Zip"].astype(str).value_counts().head(10).reset_index()
            top_pin.columns = ["Pincode", "Orders"]
            fig = px.bar(top_pin, x="Orders", y="Pincode", orientation="h",
                          title="Top 10 Pincodes by Orders", color_discrete_sequence=[PALETTE[3]])
            fig.update_layout(yaxis=dict(autorange="reversed"))
            st.plotly_chart(fig, width='stretch')

    elif sel_city == "All Cities":
        # State selected: best cities and best pincodes within that state
        col1, col2 = st.columns(2)
        with col1:
            top_cities = geo_scope["Shipping City"].value_counts().head(10).reset_index()
            top_cities.columns = ["City", "Orders"]
            fig = px.bar(top_cities, x="Orders", y="City", orientation="h",
                          title=f"Top Cities in {sel_state} by Orders", color_discrete_sequence=[ACCENT2])
            fig.update_layout(yaxis=dict(autorange="reversed"))
            st.plotly_chart(fig, width='stretch')
        with col2:
            top_pin = geo_scope["Shipping Zip"].astype(str).value_counts().head(10).reset_index()
            top_pin.columns = ["Pincode", "Orders"]
            fig = px.bar(top_pin, x="Orders", y="Pincode", orientation="h",
                          title=f"Top Pincodes in {sel_state} by Orders", color_discrete_sequence=[PALETTE[3]])
            fig.update_layout(yaxis=dict(autorange="reversed"))
            st.plotly_chart(fig, width='stretch')

    else:
        # State + city selected: best pincodes within that city
        top_pin = geo_scope["Shipping Zip"].astype(str).value_counts().head(10).reset_index()
        top_pin.columns = ["Pincode", "Orders"]
        fig = px.bar(top_pin, x="Orders", y="Pincode", orientation="h",
                      title=f"Top Pincodes in {sel_city}, {sel_state} by Orders",
                      color_discrete_sequence=[PALETTE[3]])
        fig.update_layout(yaxis=dict(autorange="reversed"))
        st.plotly_chart(fig, width='stretch')

    st.caption(f"Showing demand for {len(geo_scope):,} orders in the selected scope.")

In [ ]:
if "Order Month" in orders.columns:
    st.subheader("Seasonal / Calendar Pattern")
    cal = orders.copy()
    cal["Month Name"] = pd.to_datetime(cal["Order Date"]).dt.month_name()
    month_order = ["January", "February", "March", "April", "May", "June",
                    "July", "August", "September", "October", "November", "December"]
    cal_counts = cal["Month Name"].value_counts().reindex(month_order).reset_index()
    cal_counts.columns = ["Month", "Orders"]
    fig = px.bar(cal_counts, x="Month", y="Orders", title="Orders by Calendar Month (All Years Combined)",
                  color_discrete_sequence=[ACCENT1])
    st.plotly_chart(fig, width='stretch')

In [ ]:
st.markdown("---")
st.subheader("Order Funnel")

total_count = len(orders)
paid_count = orders["Paid at"].notna().sum() if "Paid at" in orders.columns else 0
shipped_count = orders["Is Shipped"].sum() if "Is Shipped" in orders.columns else 0
delivered_count = orders["Is Delivered"].sum() if "Is Delivered" in orders.columns else 0

funnel_df = pd.DataFrame({
    "Stage": ["Total Orders", "Paid", "Shipped", "Delivered"],
    "Orders": [total_count, paid_count, shipped_count, delivered_count],
})
fig = go.Figure(go.Funnel(
    y=funnel_df["Stage"], x=funnel_df["Orders"],
    marker=dict(color=[ACCENT1, ACCENT2, PALETTE[2], SUCCESS]),
    textinfo="value+percent initial",
))
fig.update_layout(title="Order Lifecycle Funnel")
st.plotly_chart(fig, width='stretch')
st.caption(
    "Shows where orders drop out of the lifecycle: unpaid (pending/voided), paid but never "
    "shipped, and shipped but returned (RTO) instead of delivered."
)

In [ ]:
if not paid.empty and "Total" in paid.columns:
    col1, col2 = st.columns(2)
    with col1:
        aov_trend = paid.groupby("Paid Month")["Total"].mean().reset_index()
        aov_trend["Paid Month"] = aov_trend["Paid Month"].astype(str)
        fig = px.line(aov_trend, x="Paid Month", y="Total", markers=True,
                       title="Average Order Value Trend (Paid Orders)")
        fig.update_traces(line_color=ACCENT1, fill="tozeroy", fillcolor="rgba(91,141,239,0.10)")
        fig.update_layout(yaxis_title="Avg Order Value (₹)")
        st.plotly_chart(fig, width='stretch')

    with col2:
        clip_value = paid["Total"].quantile(0.99)
        dist = paid[paid["Total"] <= clip_value]
        fig = px.histogram(dist, x="Total", nbins=40, title="Order Value Distribution (Paid Orders)",
                            color_discrete_sequence=[ACCENT2])
        fig.update_layout(xaxis_title="Order Value (₹)", yaxis_title="Orders", showlegend=False)
        st.plotly_chart(fig, width='stretch')
        st.caption(f"Clipped at the 99th percentile (₹{clip_value:,.0f}) to exclude extreme outliers.")

In [ ]:
if "Source" in orders.columns:
    st.subheader("Order Source / Channel Breakdown")
    source_counts = orders["Source"].astype(str).value_counts().head(8).reset_index()
    source_counts.columns = ["Source", "Orders"]
    fig = px.bar(source_counts, x="Orders", y="Source", orientation="h",
                  title="Top Sales Channels by Orders", color_discrete_sequence=[PALETTE[3]])
    fig.update_layout(yaxis=dict(autorange="reversed", type="category"))
    st.plotly_chart(fig, width='stretch')
    st.caption(
        "Values are Shopify sales-channel IDs (online store, draft orders, sales channel "
        "apps) rather than human-readable names -- useful for spotting concentration/shifts "
        "across channels over time."
    )